# PagedAttention: Efficient KV Cache Management

In the last notebook, we saw how requests are processed into a standard format. Now, we need to tackle one of the biggest bottlenecks in LLM inference: memory. As a model generates text token by token, it needs to store a "Key-Value cache" (or KV cache) for all previous tokens to maintain context. Managing this memory efficiently is critical for handling many users at once.

By the end of this notebook, you will understand the core concept of PagedAttention and its benefits for LLM inference. You'll see how it dramatically reduces memory waste compared to simpler methods. Finally, you will build a simplified KV cache block manager from scratch to see exactly how it works.

In [ ]:
# === Setup Cell ===
# This notebook is self-contained. All necessary components will be defined here or in the cells below.

import math
import dataclasses
from typing import List, Dict, Optional

# For our visualization later
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# A simple class to represent a block of memory in our KV Cache
@dataclasses.dataclass
class KVBlock:
    """Represents a single, fixed-size block of physical memory for the KV cache."""
    block_id: int
    block_size: int # In a real system, this would define tensor dimensions

    def __repr__(self):
        return f"Block(id={self.block_id})"

### The Core Idea: A Smarter Hotel for Your Data

Imagine you're managing a hotel. In the world of LLMs, each user request (a "sequence") is a guest who needs a room for a certain number of nights (tokens). The problem is, you don't know exactly how long each guest will stay.

**The "Contiguous" Method (The Inefficient Way):**

This is like forcing every guest to book a large, fixed-length stay—say, 30 nights—in a single, contiguous block of rooms.
*   **Problem 1: Internal Fragmentation.** If a guest only stays for 5 nights, you've wasted 25 nights' worth of room capacity that you can't give to anyone else. This is like allocating memory for the maximum possible sequence length, even for very short prompts.
*   **Problem 2: External Fragmentation.** Your hotel might have 50 rooms free in total, but if a new guest needs a block of 30 *consecutive* rooms and your largest available block is only 20, you have to turn them away. This is like having enough total memory, but not a large enough *single chunk* to satisfy a new request.

This inefficiency means you can serve fewer guests (lower throughput) and a lot of your hotel's capacity (GPU memory) goes unused.

**The PagedAttention Method (The Smart Way):**

PagedAttention works like a modern hotel that books one night at a time. A guest's stay is broken into individual nights ("pages" or "blocks").
*   A guest needing a 5-night stay gets 5 room keys. These rooms don't need to be next to each other! One could be on the 1st floor, the next on the 5th, and so on.
*   The guest has a "map" (a "page table" or "block table") that tells them: Night 1 -> Room 101, Night 2 -> Room 503, etc.
*   To the guest, their stay feels continuous. To the hotel manager, the physical rooms can be assigned from anywhere they are free.

This solves both problems. There's no internal waste because you only allocate what you need, and you can always accept a new guest as long as you have *any* free rooms, regardless of where they are. This is the core insight of PagedAttention: it allows the model's KV cache to be stored in non-contiguous, fixed-size blocks of memory, just like an operating system manages RAM.



Now, let's build a simplified version of this "hotel management" system.

In [ ]:
class SimpleKVCacheManager:
    """A simplified implementation of PagedAttention's memory management."""

    def __init__(self, num_blocks: int, block_size: int):
        """
        Initializes the manager with a pool of free memory blocks.

        Args:
            num_blocks: The total number of physical blocks available.
            block_size: The number of tokens each block can hold.
        """
        self.block_size = block_size
        # The pool of all available physical blocks
        self.all_blocks = [KVBlock(block_id=i, block_size=block_size) for i in range(num_blocks)]
        # A list of blocks that are currently not in use
        self.free_blocks = list(self.all_blocks)
        # The "page table" mapping a sequence ID to its list of allocated blocks
        self.block_table: Dict[str, List[KVBlock]] = {}

    def allocate(self, seq_id: str, num_tokens: int) -> bool:
        """Allocates blocks for a sequence."""
        num_required_blocks = math.ceil(num_tokens / self.block_size)
        print(f"INFO: Allocating for '{seq_id}' ({num_tokens} tokens)... needs {num_required_blocks} blocks.")

        if num_required_blocks > len(self.free_blocks):
            print(f"--> FAIL: Not enough free blocks. Have {len(self.free_blocks)}, need {num_required_blocks}.")
            return False

        # Take the required blocks from the free pool
        allocated_blocks = [self.free_blocks.pop(0) for _ in range(num_required_blocks)]
        self.block_table[seq_id] = allocated_blocks
        print(f"--> SUCCESS: Allocated {allocated_blocks} to '{seq_id}'.")
        return True

    def free(self, seq_id: str):
        """Frees all blocks associated with a sequence."""
        if seq_id not in self.block_table:
            return
        
        blocks_to_free = self.block_table.pop(seq_id)
        # Return the blocks to the free pool
        self.free_blocks.extend(blocks_to_free)
        # We sort to make the printout predictable, not for correctness
        self.free_blocks.sort(key=lambda b: b.block_id)
        print(f"INFO: Freed blocks for '{seq_id}': {blocks_to_free}.")

    def __repr__(self):
        return (f"KVCacheManager:\n"
                f"  Free Blocks: {len(self.free_blocks)} -> {self.free_blocks}\n"
                f"  Allocated Table: {self.block_table}")

### Walking Through the Implementation

Let's break down our `SimpleKVCacheManager`. It has three key parts that mirror the PagedAttention concept.

1.  **`__init__(self, num_blocks, block_size)`**: This is our "hotel construction" phase.
    *   `self.all_blocks`: We create our total physical memory, a simple list of `KVBlock` objects. Each block has a unique `block_id`.
    *   `self.free_blocks`: This is the list of currently available "rooms". Initially, all blocks are free.
    *   `self.block_table`: This is the crucial "map" or "page table". It's a dictionary where keys are the sequence IDs (our "guests") and values are the list of `KVBlock`s they are using. This table provides the logical-to-physical mapping.

2.  **`allocate(self, seq_id, num_tokens)`**: This is the "check-in" process.
    *   `num_required_blocks = math.ceil(num_tokens / self.block_size)`: We calculate how many blocks (rooms) the sequence (guest) needs based on its length. We use `math.ceil` because even if a sequence needs just one token's worth of space in a new block, it must occupy the whole block.
    *   We check if `len(self.free_blocks)` is sufficient. If not, the allocation fails.
    *   `allocated_blocks = [self.free_blocks.pop(0) ... ]`: We take the required number of blocks from our pool of free blocks. Notice that we just take the first ones available—we don't care about their `block_id` or physical location.
    *   `self.block_table[seq_id] = allocated_blocks`: We create an entry in our map, linking the `seq_id` to the specific physical blocks it now owns.

3.  **`free(self, seq_id)`**: This is the "check-out" process.
    *   `blocks_to_free = self.block_table.pop(seq_id)`: We find the guest in our map and get the list of rooms they were using. We then remove them from the map.
    *   `self.free_blocks.extend(blocks_to_free)`: We return those blocks to the pool of available rooms, ready to be used by the next guest.

In [ ]:
# === Demonstration ===

# Initialize a small KV Cache with 10 total blocks, each holding 4 tokens.
kv_cache = SimpleKVCacheManager(num_blocks=10, block_size=4)
print("--- Initial State ---")
print(kv_cache)

# 1. Request A arrives, it's a prompt of 7 tokens.
# It will need ceil(7 / 4) = 2 blocks.
print("\n--- Request A arrives (7 tokens) ---")
kv_cache.allocate(seq_id="request_A", num_tokens=7)
print(kv_cache)

# 2. Request B arrives, it's a prompt of 10 tokens.
# It will need ceil(10 / 4) = 3 blocks.
print("\n--- Request B arrives (10 tokens) ---")
kv_cache.allocate(seq_id="request_B", num_tokens=10)
print(kv_cache)

# 3. Request A finishes generation and is released.
# Its 2 blocks should be returned to the free pool.
print("\n--- Request A finishes ---")
kv_cache.free(seq_id="request_A")
print(kv_cache)

# 4. Request C arrives, it's a large prompt of 15 tokens.
# It will need ceil(15 / 4) = 4 blocks.
# It can reuse the blocks just freed by Request A!
print("\n--- Request C arrives (15 tokens) ---")
kv_cache.allocate(seq_id="request_C", num_tokens=15)
print(kv_cache)

# 5. Request D arrives, needing 20 tokens (5 blocks)
# We only have 10 - 3 (B) - 4 (C) = 3 blocks free. It should fail.
print("\n--- Request D arrives (20 tokens) ---")
kv_cache.allocate(seq_id="request_D", num_tokens=20)
print(kv_cache)

### Going Deeper: Sharing is Caring (Prefix Caching)

One of the most powerful features enabled by PagedAttention is **prefix caching**. Imagine 100 users all starting their conversation with your chatbot by asking, "What is the capital of France?".

It would be incredibly wasteful to compute and store the KV cache for this common prompt 100 separate times. With PagedAttention, we can be much smarter. Since memory is managed at the block level, we can have multiple sequences *point to the same physical blocks* for their shared prefix.

To do this, we need to add a **reference counter** to each block.
*   When a block is assigned to a sequence, its reference count is incremented.
*   When a sequence frees a block, its reference count is decremented.
*   A block is only returned to the `free_blocks` pool when its reference count hits zero.

Let's build a slightly more advanced manager to see this in action. We will add a `clone` method to simulate a new request sharing the prefix of an existing one.

In [ ]:
class AdvancedKVCacheManager:
    """A manager that supports prefix sharing via reference counting."""

    def __init__(self, num_blocks: int, block_size: int):
        self.block_size = block_size
        self.all_blocks = [KVBlock(block_id=i, block_size=block_size) for i in range(num_blocks)]
        self.free_blocks = list(self.all_blocks)
        self.block_table: Dict[str, List[KVBlock]] = {}
        # NEW: Track the reference count of each physical block
        self.ref_counts: Dict[int, int] = {i: 0 for i in range(num_blocks)}

    def _increment_ref_counts(self, blocks: List[KVBlock]):
        for block in blocks:
            self.ref_counts[block.block_id] += 1
            
    def _decrement_ref_counts(self, blocks: List[KVBlock]):
        for block in blocks:
            self.ref_counts[block.block_id] -= 1
            # If no sequence is using this block anymore, it's free
            if self.ref_counts[block.block_id] == 0:
                self.free_blocks.append(block)

    def allocate(self, seq_id: str, num_tokens: int):
        num_required_blocks = math.ceil(num_tokens / self.block_size)
        if num_required_blocks > len(self.free_blocks):
            return False # Not enough memory

        allocated_blocks = [self.free_blocks.pop(0) for _ in range(num_required_blocks)]
        self.block_table[seq_id] = allocated_blocks
        self._increment_ref_counts(allocated_blocks)
        print(f"Allocated {allocated_blocks} for '{seq_id}'.")
        return True

    def clone(self, new_seq_id: str, existing_seq_id: str):
        """Creates a new sequence that shares the blocks of an existing one."""
        if existing_seq_id not in self.block_table:
            print(f"Cannot clone '{existing_seq_id}', it does not exist.")
            return

        shared_blocks = self.block_table[existing_seq_id]
        self.block_table[new_seq_id] = list(shared_blocks) # Make a copy of the list
        self._increment_ref_counts(shared_blocks)
        print(f"Cloned '{existing_seq_id}' to '{new_seq_id}'. Both share {shared_blocks}.")

    def free(self, seq_id: str):
        if seq_id not in self.block_table:
            return
        
        blocks_to_free = self.block_table.pop(seq_id)
        self._decrement_ref_counts(blocks_to_free)
        self.free_blocks.sort(key=lambda b: b.block_id)
        print(f"Freed '{seq_id}'.")

    def __repr__(self):
         return (f"KVCacheManager:\n"
                f"  Free Blocks: {len(self.free_blocks)}\n"
                f"  Ref Counts: {self.ref_counts}\n"
                f"  Block Table: {self.block_table}")

# --- Demonstration of Prefix Caching ---
adv_cache = AdvancedKVCacheManager(num_blocks=8, block_size=4)
print("--- Initial State ---")
print(adv_cache)

print("\n--- User 1 sends a prompt (10 tokens) ---")
adv_cache.allocate(seq_id="user_1_prompt", num_tokens=10) # Needs 3 blocks
print(adv_cache)

print("\n--- User 2 sends the SAME prompt ---")
adv_cache.clone(new_seq_id="user_2_prompt", existing_seq_id="user_1_prompt")
print("Note: No new blocks were taken from the free pool!")
print(adv_cache)

print("\n--- User 1's request finishes ---")
adv_cache.free(seq_id="user_1_prompt")
print("Note: The blocks are NOT returned to the free pool, because User 2 still needs them.")
print(adv_cache)

print("\n--- User 2's request finishes ---")
adv_cache.free(seq_id="user_2_prompt")
print("Note: NOW the blocks are returned, as their ref_count hit 0.")
print(adv_cache)

### Visualization: Logical vs. Physical Memory

The text-based output is helpful, but a visualization can make the benefit of PagedAttention crystal clear.

The plot below shows two views of our KV cache:
*   **Logical View (top):** This is how each sequence *sees* its own memory. To `request_A`, its tokens live in a simple, continuous array from 0 to 8. It doesn't know or care about the underlying complexity.
*   **Physical View (bottom):** This is the ground truth of our GPU memory. It's a grid of all available blocks.

We will simulate a few requests arriving and leaving, and color-code the blocks to see how the logical sequences are mapped to the physical blocks. Pay close attention to how `request_C` reuses the fragmented space left behind by `request_A`. This is something a contiguous allocator could not do!

In [ ]:
def visualize_cache_state(manager: SimpleKVCacheManager, title: str):
    """Creates a plot to visualize the logical-to-physical mapping."""
    
    num_blocks = len(manager.all_blocks)
    block_size = manager.block_size
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), gridspec_kw={'height_ratios': [1, 2]})
    fig.suptitle(title, fontsize=16)

    # --- 1. Logical View ---
    ax1.set_title("Logical View (Per-Sequence)")
    ax1.set_xlim(-1, num_blocks * block_size)
    ax1.set_ylim(-0.5, len(manager.block_table) + 0.5)
    ax1.set_yticks(range(len(manager.block_table)))
    ax1.set_yticklabels(list(manager.block_table.keys()))
    ax1.set_xlabel("Token Position")

    colors = plt.cm.get_cmap('viridis', len(manager.block_table))
    seq_colors = {seq_id: colors(i) for i, seq_id in enumerate(manager.block_table.keys())}

    for i, (seq_id, blocks) in enumerate(manager.block_table.items()):
        num_tokens = len(blocks) * block_size # For simplicity, assume all blocks are full
        ax1.add_patch(patches.Rectangle((0, i-0.4), num_tokens, 0.8, facecolor=seq_colors[seq_id], alpha=0.7))
        ax1.text(1, i, f"{num_tokens} tokens")

    # --- 2. Physical View ---
    ax2.set_title("Physical View (Actual Memory Blocks)")
    ax2.set_xlim(-0.5, num_blocks + 0.5)
    ax2.set_ylim(0, 2)
    ax2.set_xticks(range(num_blocks))
    ax2.set_xticklabels([f"Block {i}" for i in range(num_blocks)], rotation=45, ha="right")
    ax2.set_yticks([])

    physical_block_map = {block.block_id: "free" for block in manager.all_blocks}
    for seq_id, blocks in manager.block_table.items():
        for block in blocks:
            physical_block_map[block.block_id] = seq_id

    for i in range(num_blocks):
        status = physical_block_map[i]
        color = 'lightgray'
        if status != "free":
            color = seq_colors[status]
        
        ax2.add_patch(patches.Rectangle((i - 0.4, 0.5), 0.8, 0.8, facecolor=color, edgecolor='black'))
        ax2.text(i, 1.5, status.replace("request_", ""), ha='center', va='center', fontsize=9)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# --- Visualization Simulation ---
vis_cache = SimpleKVCacheManager(num_blocks=16, block_size=4)

# 1. Request A arrives (9 tokens -> 3 blocks)
vis_cache.allocate("request_A", 9)
# 2. Request B arrives (6 tokens -> 2 blocks)
vis_cache.allocate("request_B", 6)
visualize_cache_state(vis_cache, "State 1: Requests A and B are active")

# 3. Request A finishes, freeing its blocks
vis_cache.free("request_A")
visualize_cache_state(vis_cache, "State 2: Request A is freed, leaving fragmented memory")

# 4. Request C arrives (15 tokens -> 4 blocks)
vis_cache.allocate("request_C", 15)
visualize_cache_state(vis_cache, "State 3: Request C re-uses the fragmented blocks")

### Summary

In this notebook, we demystified PagedAttention, one of the key technologies that makes high-throughput LLM serving possible.

**What We Built:**
We created a `SimpleKVCacheManager` that simulates the core logic of PagedAttention. It manages a pool of fixed-size memory blocks, using a "block table" to map them to sequences. This demonstrates how logically contiguous memory for a sequence can be composed of physically non-contiguous blocks. We also extended this to an `AdvancedKVCacheManager` that supports prefix caching via reference counting.

**Key Takeaways:**
*   **Virtual vs. Physical Memory:** PagedAttention separates the logical view of a sequence's memory from the physical layout of the GPU's memory.
*   **Eliminating Fragmentation:** It solves both internal fragmentation (by not over-allocating within a block) and external fragmentation (by using any free block, anywhere).
*   **Higher Throughput:** By using memory far more efficiently (often achieving >90% utilization), a system like vLLM can handle many more concurrent requests on the same hardware.
*   **Enabling New Features:** This block-based management naturally enables powerful features like prefix caching and sharing, further boosting efficiency.

**What's Next:**
Now that we have a solid understanding of how the KV cache memory is managed, the next question is: how does the model *use* this information? In the next notebook, **The LLM Model Executor**, we will see how the block table is passed to the GPU kernels to perform the actual attention calculations on this paged, non-contiguous data.